# Multi-Object Tracking Under Occlusion (AIML-03)
### End-to-End Pipeline Walkthrough & Experimental Analysis

This interactive notebook demonstrates the fundamental computer vision and state estimation components of the **SORT (Simple Online and Realtime Tracking)** system designed for ceiling cameras in retail environments.

In [ ]:
import sys, os
# Ensure project root is accessible
PROJECT_ROOT = os.path.abspath("..")
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import cv2
import numpy as np
import matplotlib.pyplot as plt
from ultralytics.utils import ASSETS

from src.detector.yolo_detector import YOLODetector
from src.tracker.association import iou, associate_detections_to_trackers
from src.tracker.kalman_filter import KalmanBoxTrackerFilter
from src.tracker.sort import Sort
from src.visualization.visualize import Visualizer

print("All core tracking modules loaded successfully!")

## 1. YOLO Person Detection
Using Ultralytics YOLO with COCO pre-trained weights, filtered exclusively for **Class 0 (Person)**.
Internal detection format: `[x1, y1, x2, y2, confidence]`

In [ ]:
detector = YOLODetector(model_name="yolov8n.pt", conf_threshold=0.35)
bus_image = cv2.imread(os.path.join(ASSETS, "bus.jpg"))

detections = detector.detect(bus_image)
print(f"Detected {len(detections)} persons:")
for i, det in enumerate(detections):
    print(f"  Person #{i+1}: [{det[0]:.1f}, {det[1]:.1f}, {det[2]:.1f}, {det[3]:.1f}], Conf: {det[4]:.3f}")

annotated = detector.draw_detections(bus_image, detections)
plt.figure(figsize=(10, 6))
plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
plt.title("YOLOv8 Person Detections")
plt.axis("off")
plt.show()

## 2. Intersection over Union (IoU) & Hungarian Assignment
The Hungarian algorithm finds the optimal global matching between predicted track positions and new detections.

In [ ]:
box_a = np.array([50, 50, 150, 200])
box_b = np.array([70, 70, 170, 220])
score = iou(box_a, box_b)
print(f"IoU overlap between Box A and Box B: {score:.4f}")

# Sample Hungarian Matching
dets = np.array([
    [52, 52, 152, 202, 0.92],
    [300, 100, 380, 250, 0.88]
])
trks = np.array([
    [50, 50, 150, 200],
    [600, 100, 680, 250]
])
matches, unmatched_dets, unmatched_trks = associate_detections_to_trackers(dets, trks, iou_threshold=0.3)
print("Hungarian Matches [Detection Index -> Tracker Index]:", matches.tolist())
print("Unmatched Detections (New Identifies):", unmatched_dets.tolist())
print("Unmatched Trackers (Occluded Targets):", unmatched_trks.tolist())

## 3. Kalman Filter: Prediction Under Occlusion
Demonstrating that when measurements stop arriving for 4 consecutive frames, the Kalman filter predicts the physical forward trajectory without losing track.

In [ ]:
initial_box = np.array([100, 100, 150, 250], dtype=np.float32)
kf = KalmanBoxTrackerFilter(initial_box)

# Initial walking observed
for f in range(1, 6):
    measurement = initial_box + np.array([f * 10, 0, f * 10, 0])
    kf.predict()
    kf.update(measurement)

print("--- Beginning 4 Frames of Occlusion (No Detections) ---")
for f in range(6, 10):
    pred = kf.predict()
    print(f"Frame {f} (Occluded): Predicted Center-X = {pred[0] + 25:.1f} px | True Physical X = {125 + f * 10:.1f} px")

## 4. End-to-End Tracking on Retail Occlusion Sequence
Evaluating SORT on the synthetic retail sequence with standard CLEAR MOT metrics.

In [ ]:
from src.evaluation.evaluate import evaluate_tracker_on_sequence

tracker = Sort(max_age=30, min_hits=2, iou_threshold=0.3)
metrics = evaluate_tracker_on_sequence("../data/sample/retail_seq", detector, tracker, min_visibility=0.1)

print("CLEAR MOT Evaluation Summary:")
for k, v in metrics.items():
    print(f"  {k:22s}: {v}")

## 5. Visualizing Trajectories & Diagnostic Failure Analysis

In [ ]:
fig_path = "../results/figures/occlusion_case_study.png"
if os.path.exists(fig_path):
    img = cv2.imread(fig_path)
    plt.figure(figsize=(16, 7))
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.show()

breakdown_path = "../results/figures/failure_mode_breakdown.png"
if os.path.exists(breakdown_path):
    img2 = cv2.imread(breakdown_path)
    plt.figure(figsize=(10, 6))
    plt.imshow(cv2.cvtColor(img2, cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.show()